# Supplementary notebook 4. Where the dual problem is solved (Fig. S7)

Reproduces SI Fig. S7. Reads no data. Run time about 14 minutes. `QUICK = True` in the first code cell shortens the runs for a test.

In [1]:
QUICK = False
T_RUN = 3000      # the run read by Fig. 4 and the copying analysis of Fig. S2
T_POP = 5000      # the four conditions, the hypothesis tests, the sweeps and Fig. 5; the same seeds,
                  # so the first 3,000 steps of these runs are the runs of Fig. 4

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from math import log2

plt.rcParams.update({'figure.dpi': 110, 'font.size': 8, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.7,
                     'xtick.major.width': 0.7, 'ytick.major.width': 0.7,
                     'legend.frameon': False, 'font.family': 'DejaVu Sans'})

CEIL = log2(100)          # 6.6439 bits, the even-coupling ceiling at N = 100
BA_KAPPA = 10.10          # matched scale-free ensemble, control-entropy paper Table 1
BA_DH_DEG = 0.357         # its degree-weighted compression
BA_RDEG = -0.17           # its degree assortativity
N = 100
COL = {'baseline': '#7A7A7A', 'explore': '#2E7D4E', 'imitate': '#2E5090', 'both': '#A4161A'}
CONDS = {'baseline': dict(pr=0.0, pe=0.0), 'explore': dict(pr=0.0, pe=0.05),
         'imitate': dict(pr=0.3, pe=0.0), 'both': dict(pr=0.3, pe=0.05)}

def seeds(n):
    return list(range(n if not QUICK else max(2, n // 5)))

print('ceiling log2 N =', round(CEIL, 4), 'bits')

ceiling log2 N = 6.6439 bits


## Step 0. The model

In [2]:
import numpy as np
import networkx as nx

LOG2 = np.log(2.0)


# ----------------------------------------------------------------------
# measures (definitions of the second study)
# ----------------------------------------------------------------------
def control_entropy(v):
    """Control entropy in bits of a non-negative vector of shares (Eq. 11)."""
    v = np.asarray(v, float)
    tot = v.sum()
    if tot <= 0:
        return 0.0
    p = v / tot
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def eigen_shares(W):
    """Shares from the leading eigenvector of a symmetric coupling matrix."""
    vals, vecs = np.linalg.eigh(W)
    v = np.abs(vecs[:, -1])
    return v / v.sum()


def adjacency_measures(A, W):
    """Measures read from the unweighted structure, computed as in Table 1 of the
    control-entropy paper: eigenvector weighting on the largest connected component,
    degree assortativity on the whole graph."""
    G = nx.from_numpy_array(A.astype(int))
    deg = A.sum(axis=1)
    s = W.sum(axis=1)
    iu, ju = np.nonzero(np.triu(A, 1))

    def edge_corr(v):
        x, y = np.concatenate([v[iu], v[ju]]), np.concatenate([v[ju], v[iu]])
        return float(np.corrcoef(x, y)[0, 1]) if x.std() > 1e-12 else float('nan')

    r_deg, r_str = edge_corr(deg.astype(float)), edge_corr(s)
    if not nx.is_connected(G):
        G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    c = nx.eigenvector_centrality_numpy(G)
    v = np.abs(np.array([c[n] for n in G.nodes()]))
    return dict(H_adj=control_entropy(v), r_deg=float(r_deg), r_str=r_str, gc=G.number_of_nodes())


# ----------------------------------------------------------------------
# the five rules, one function each
# ----------------------------------------------------------------------
def ring_lattice(N, k):
    """Regular ring lattice: every element coupled to its k nearest neighbours."""
    return nx.to_numpy_array(nx.watts_strogatz_graph(N, k, 0.0), dtype=bool)


def softmax_allocation(A, beta, shat):
    """Eq 1. Element i splits one unit of coupling over its partners j with weights
    exp(beta_i * shat_j). beta_i = 0 gives even coupling."""
    L = np.where(A, beta[:, None] * shat[None, :], -np.inf)
    L = L - L.max(axis=1, keepdims=True)
    E = np.where(A, np.exp(L), 0.0)
    return E / E.sum(axis=1, keepdims=True)


def commitment(phi0, phi1, C, fm):
    """Eq 2. Commitment of an element, in the linear form of the published model,
    f = phi0 + phi1 * C, truncated to [0, 1 - fm]."""
    return np.clip(phi0 + phi1 * C, 0.0, 1.0 - fm)


def protection(f, c, pmax, chalf):
    """Eq 8. Saturating protection bought by committed capability f * c."""
    x = f * c
    return np.where(x > 0, pmax / (1.0 + chalf / np.maximum(x, 1e-12)), 0.0)


def fermi(dc, s):
    """Eq 10. Probability of copying a partner whose capability is higher by dc."""
    return 1.0 / (1.0 + np.exp(-s * dc))


def logistic(z, g, theta):
    """Response function of an element, used inside the spread (Eq 5)."""
    return 1.0 / (1.0 + np.exp(-g * (z - theta)))


class ElementModel:
    """Network-agent model of element-level exploration and imitation.

    Each element holds
      beta_i        direction of coupling (0 = even coupling, > 0 = toward stronger partners)
      phi0_i, phi1_i commitment strategy, giving f_i = phi0_i + phi1_i * C_i
      c_i           capability, the accumulated stimulation it compares with partners
    Each coupling (i, j) holds a weight w_ij.
    """

    def __init__(self, N=100, k=6, seed=0,
                 pr=0.3, pe=0.05, sigma=0.1, sigma_f=0.01, s=2.0,   # strategy dynamics
                 M=10, rho=0.5, nE=5, switch=None, psi=0.3,        # feedback, two channels
                 gamma=1.0, pn=0.02, pl=0.3, lam=1.0, g=4.0, theta=1.0,   # seeding and spread
                 fm=0.1, pmax=1.0, chalf=0.1, f0=0.1,              # capability and protection
                 eta=0.05, delta=0.01, w0=1.0,                     # consolidation
                 pw=2.0, kmin=2, loss=1.0,                         # pruning and its cost
                 spread=True, two_part=True):
        self.p = dict(N=N, k=k, pr=pr, pe=pe, sigma=sigma, sigma_f=sigma_f, s=s, M=M, rho=rho,
                      nE=nE, switch=switch, psi=psi, gamma=gamma, pn=pn, pl=pl, lam=lam, g=g,
                      theta=theta, fm=fm, pmax=pmax, chalf=chalf, f0=f0, eta=eta, delta=delta,
                      w0=w0, pw=pw, kmin=kmin, loss=loss, spread=spread, two_part=two_part)
        self.rng = np.random.default_rng(seed)
        self.N, self.k = N, k
        self.A = ring_lattice(N, k)
        self.W = self.A.astype(float) * w0
        self.beta = np.zeros(N)
        self.phi0 = np.full(N, f0)
        self.phi1 = np.zeros(N)
        self.c = np.ones(N)
        self.x = np.zeros(N)          # activation of the previous step
        self.v = np.ones(N) / np.sqrt(N)   # leading eigenvector, kept warm
        self.t = 0
        self.n_act = 0.0
        self.n_rounds = 0.0
        self.set_essential(self.rng.choice(N, nE, replace=False))

    # ---------- state helpers ----------
    def set_essential(self, E):
        self.E = np.asarray(E)
        self.isE = np.zeros(self.N, bool)
        self.isE[self.E] = True

    def strengths(self):
        return self.W.sum(axis=1)

    def shat(self):
        s = self.strengths()
        return s / s.max()

    def centrality(self, iters=8):
        """Leading eigenvector of W by power iteration, warm started, scaled to a
        maximum of one. This is the C of the commitment rule (Eq 2)."""
        v = self.v
        for _ in range(iters):
            v = self.W @ v
            n = np.linalg.norm(v)
            if n < 1e-12:
                v = np.ones(self.N) / np.sqrt(self.N)
                break
            v = v / n
        self.v = np.abs(v)
        return self.v / max(self.v.max(), 1e-12)

    def allocation(self):
        return softmax_allocation(self.A, self.beta, self.shat())

    # ---------- one step ----------
    def feedback(self):
        """Eq 3. Two channels of information.
        Process feedback is intrinsic: every element receives afferent input in
        proportion to its own activation in the previous step.
        Result feedback is augmented: M packets, each landing on a task-essential
        element with probability rho and on any element otherwise."""
        p, rng = self.p, self.rng
        u = p['psi'] * self.x.copy()
        if p['M'] > 0:
            to_E = rng.random(p['M']) < p['rho']
            tgt = np.where(to_E, rng.choice(self.E, p['M']), rng.integers(0, self.N, p['M']))
            u = u + np.bincount(tgt, minlength=self.N).astype(float)
        return u

    def spread(self, u, a):
        """Eq 4 and Eq 5. Stimulus is seeded where information arrives and then
        travels along the couplings.
        Seeding: element i is stimulated with probability 1 - (1 - p_n) exp(-gamma u_i),
        so an element fires spontaneously at rate p_n and more often when information
        reaches it.
        Spread: an element already stimulated passes activation to partner j with
        the coupling it allocates, and j responds with probability
        pl * logistic(g (d_j - theta)). Each element activates at most once per step.
        The cascade is run to completion inside the step."""
        p, rng, N = self.p, self.rng, self.N
        act = rng.random(N) < (1.0 - (1.0 - p['pn']) * np.exp(-p['gamma'] * u))   # Eq 4
        if not p['spread']:
            self.n_act, self.n_rounds = float(act.sum()), 0.0
            return act
        frontier = act.copy()
        rounds = 0
        while frontier.any() and rounds < 20:
            # drive reaching each candidate through the allocation of active elements
            d = p['lam'] * self.k * (frontier @ a)
            prob = p['pl'] * logistic(d, p['g'], p['theta'])       # Eq 5
            new = (rng.random(N) < prob) & ~act
            act |= new
            frontier = new
            rounds += 1
        self.n_act, self.n_rounds = float(act.sum()), float(rounds)
        return act

    def step(self):
        p, rng, N = self.p, self.rng, self.N
        if p['switch'] and self.t > 0 and self.t % p['switch'] == 0:
            self.set_essential(rng.choice(N, p['nE'], replace=False))

        C = self.centrality()
        a = self.allocation()                                     # Eq 1
        f = commitment(self.phi0, self.phi1, C, p['fm']) if p['two_part'] else np.zeros(N)  # Eq 2
        u = self.feedback()                                       # Eq 3
        act = self.spread(u, a)                                   # Eq 4, Eq 5
        x = act.astype(float)
        self.x = x

        # Eq 6. Payoff is the stimulation the element gathers. It reaches partners
        # through its own allocation and it integrates in proportion to the coupling
        # it holds, so both the direction and the amount of coupling enter.
        s = self.strengths()
        pay = (s / s.mean()) * (a @ x)
        self.c = pay + np.clip(1.0 - p['fm'] - f, 0.0, None) * self.c   # Eq 7

        # Eq 8 and Eq 9. Commitment protects a coupling against decay.
        P = protection(f, self.c, p['pmax'], p['chalf'])
        Pij = 0.5 * (P[:, None] + P[None, :])
        abar = 0.5 * (a + a.T)
        self.W = np.where(self.A,
                          p['w0'] + (1 - p['delta'] * (1 - Pij)) * (self.W - p['w0'])
                          + p['eta'] * self.k * abar * np.outer(x, x), 0.0)

        # Eq S1. Long-term change: a coupling falls out of use, and unless the
        # capability committed to it holds it, it is lost and replaced elsewhere.
        n_rw = int(p['pw']) + (1 if rng.random() < p['pw'] - int(p['pw']) else 0)
        for _ in range(n_rw):
            self._rewire(P)

        # Eq 10. Imitation along a coupling, synchronous, two independent passes.
        b_old, f0_old, f1_old, c_old = self.beta.copy(), self.phi0.copy(), self.phi1.copy(), self.c.copy()
        for i in np.flatnonzero(rng.random(N) < p['pr']):
            nb = np.flatnonzero(self.A[i])
            j = nb[rng.integers(nb.size)]
            if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                self.beta[i] = b_old[j]
        if p['two_part']:
            for i in np.flatnonzero(rng.random(N) < p['pr']):
                nb = np.flatnonzero(self.A[i])
                j = nb[rng.integers(nb.size)]
                if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                    self.phi0[i], self.phi1[i] = f0_old[j], f1_old[j]

        # Eq S2. Exploration of both parts of the strategy.
        ex = rng.random(N) < p['pe']
        self.beta[ex] += rng.normal(0.0, p['sigma'], ex.sum())
        if p['two_part']:
            ex0 = rng.random(N) < p['pe']
            self.phi0[ex0] += rng.normal(0.0, p['sigma_f'], ex0.sum())
            ex1 = rng.random(N) < p['pe']
            self.phi1[ex1] += rng.normal(0.0, p['sigma_f'], ex1.sum())
        self.t += 1
        return x

    def _rewire(self, P):
        """One element drops its weakest coupling, unless protection holds it, and
        attaches to a new partner chosen with exp(beta_i * shat_j)."""
        p, rng = self.p, self.rng
        i = rng.integers(self.N)
        deg = self.A.sum(axis=1)
        nb = np.flatnonzero(self.A[i])
        nb = nb[deg[nb] > p['kmin']]
        if deg[i] <= p['kmin'] or nb.size == 0:
            return
        j = nb[np.argmin(self.W[i, nb])]
        if rng.random() < 0.5 * (P[i] + P[j]):     # the coupling is held
            return
        # the route is lost, and with it the capability the two elements had built
        # through it. With loss = 1 this is the capability loss of the published model.
        self.c[i] *= (1.0 - p['loss'])
        self.c[j] *= (1.0 - p['loss'])
        cand = np.flatnonzero(~self.A[i])
        cand = cand[cand != i]
        if cand.size == 0:
            return
        z = self.beta[i] * self.shat()[cand]
        w = np.exp(z - z.max())
        new = rng.choice(cand, p=w / w.sum())
        self.A[i, j] = self.A[j, i] = False
        self.W[i, j] = self.W[j, i] = 0.0
        self.A[i, new] = self.A[new, i] = True
        self.W[i, new] = self.W[new, i] = p['w0']

    # ---------- measures ----------
    def measures(self, adj=False):
        s = self.strengths()
        deg = self.A.sum(axis=1)
        top = np.argsort(s)[-len(self.E):]
        nbE = self.A[self.E].any(axis=0) & ~self.isE
        tot = s.sum()
        C = self.centrality()
        f = commitment(self.phi0, self.phi1, C, self.p['fm'])
        return dict(H_str=control_entropy(s), H_eig=control_entropy(eigen_shares(self.W)),
                    H_deg=control_entropy(deg), kappa=(deg ** 2).mean() / deg.mean(),
                    beta_mean=self.beta.mean(), beta_var=self.beta.var(),
                    f_mean=float(f.mean()), phi0_mean=float(self.phi0.mean()),
                    phi1_mean=float(self.phi1.mean()), c_mean=float(self.c.mean()),
                    act=self.n_act / self.N, rounds=self.n_rounds,
                    hub_align=self.isE[top].mean(),
                    share_E=s[self.isE].sum() / tot, share_nbE=s[nbE].sum() / tot,
                    **(adjacency_measures(self.A, self.W) if adj else {}))

    def run(self, T, every=25, adj=False):
        rec = {}
        for step in range(T):
            self.step()
            if step % every == 0 or step == T - 1:
                m = self.measures(adj=adj)
                m['t'] = self.t
                for key, v in m.items():
                    rec.setdefault(key, []).append(v)
        return {key: np.array(v) for key, v in rec.items()}


def replicate(T, seeds, adj=True, **kw):
    """Run the model for several seeds and return the final measures."""
    out = []
    for sd in seeds:
        m = ElementModel(seed=sd, **kw)
        for _ in range(T):
            m.step()
        out.append(m.measures(adj=adj))
    return {key: np.array([o[key] for o in out]) for key in out[0]}

## Step 1. The map over precision and exploration

In [3]:
RHO2 = [0.0, 0.25, 0.5, 0.75, 1.0]
PE2 = [0.0, 0.01, 0.02, 0.05, 0.1, 0.2]
DS = seeds(5)
DUAL = np.zeros((len(RHO2), len(PE2)))
HUB = np.zeros_like(DUAL)
for i, rho in enumerate(RHO2):
    for j, pe in enumerate(PE2):
        r = replicate(T_POP, DS, adj=False, rho=rho, pe=pe, pr=0.3)
        DUAL[i, j] = (CEIL - r['H_str']).mean()
        HUB[i, j] = r['share_E'].mean()
print(f'{len(DS)} seeds per cell')
print('compression (rows: rho =', RHO2, ', columns: p_e =', PE2, ')')
for i, rho in enumerate(RHO2):
    print(f'rho = {rho:<5} ' + ' '.join(f'{v:5.2f}' for v in DUAL[i]))
print('share on essential elements (same grid)')
for i, rho in enumerate(RHO2):
    print(f'rho = {rho:<5} ' + ' '.join(f'{v:5.3f}' for v in HUB[i]))
print(f'largest compression on the grid {DUAL.max():.2f} bits, at rho = {RHO2[np.argmax(DUAL) // len(PE2)]}, '
      f'p_e = {PE2[np.argmax(DUAL) % len(PE2)]}')
print(f'largest essential share on the grid {HUB.max():.3f}')
print(f'largest compression with no aimed information (rho = 0): {DUAL[0].max():.2f} bits, '
      f'essential share never above {HUB[0].max():.3f}')
print(f'largest compression with no exploration (p_e = 0): {DUAL[:, 0].max():.2f} bits, '
      f'essential share never above {HUB[:, 0].max():.3f}')
print(f'model setting rho = 0.5, p_e = 0.05: compression {DUAL[2, 3]:.2f} bits, essential share {HUB[2, 3]:.3f}')

5 seeds per cell
compression (rows: rho = [0.0, 0.25, 0.5, 0.75, 1.0] , columns: p_e = [0.0, 0.01, 0.02, 0.05, 0.1, 0.2] )
rho = 0.0    0.40  0.47  0.40  0.67  0.62  1.06
rho = 0.25   0.48  0.54  0.63  0.96  1.08  0.61
rho = 0.5    0.65  0.34  0.74  0.96  1.25  1.32
rho = 0.75   0.71  0.85  0.34  1.15  1.10  1.57
rho = 1.0    0.84  0.69  0.81  1.14  1.24  1.53
share on essential elements (same grid)
rho = 0.0   0.049 0.041 0.045 0.050 0.036 0.037
rho = 0.25  0.159 0.177 0.220 0.318 0.383 0.242
rho = 0.5   0.226 0.181 0.328 0.368 0.431 0.453
rho = 0.75  0.256 0.354 0.199 0.421 0.396 0.465
rho = 1.0   0.288 0.326 0.342 0.436 0.455 0.499
largest compression on the grid 1.57 bits, at rho = 0.75, p_e = 0.2
largest essential share on the grid 0.499
largest compression with no aimed information (rho = 0): 1.06 bits, essential share never above 0.050
largest compression with no exploration (p_e = 0): 0.84 bits, essential share never above 0.288
model setting rho = 0.5, p_e = 0.05: compression 

## Step 2. The drift of the direction preference against its own value

In [4]:
def drift_trace(T=T_POP, seed=0, **kw):
    m = ElementModel(seed=seed, **kw)
    def f_now():
        v = m.v.copy(); C = m.centrality(); m.v = v      # read the centrality without touching the state
        return commitment(m.phi0, m.phi1, C, m.p['fm']).mean()
    bb, ff = [m.beta.mean()], [f_now()]
    for _ in range(T):
        m.step()
        bb.append(m.beta.mean()); ff.append(f_now())
    return np.array(bb), np.array(ff)

TR_B, TR_F = [], []
for sd in seeds(30):
    b, f = drift_trace(seed=sd)
    TR_B.append(b); TR_F.append(f)
ends = np.array([b[-1] for b in TR_B])
print(f'{len(TR_B)} runs traced. Final mean preference per run:', ' '.join(f'{e:+.2f}' for e in ends))
print(f'median final preference {np.median(ends):.2f}, range {ends.min():+.2f} to {ends.max():+.2f}, '
      f'{int((ends > 2).sum())} runs above 2 and {int((ends < 0.5).sum())} below 0.5')

def phase(traces, nbin=14):
    x = np.concatenate([t[:-1] for t in traces])
    dx = np.concatenate([np.diff(t) for t in traces])
    lo, hi = np.percentile(x, [1, 99])
    edges = np.linspace(lo, hi, nbin + 1)
    idx = np.clip(np.digitize(x, edges) - 1, 0, nbin - 1)
    mid = 0.5 * (edges[:-1] + edges[1:])
    mean = np.array([dx[idx == i].mean() if (idx == i).any() else np.nan for i in range(nbin)])
    sem = np.array([dx[idx == i].std(ddof=1) / max(1, np.sqrt((idx == i).sum()))
                    if (idx == i).sum() > 1 else np.nan for i in range(nbin)])
    return mid, mean, sem

mb, db, sb = phase(TR_B)
mf, df_, sf = phase(TR_F)
print('drift of the mean direction preference over the range the runs reach (per step)')
for a, b, c in zip(mb, db, sb):
    print(f'  beta {a:+.2f}: {b:+.5f} +/- {c:.5f}')
lo_bin, hi_bin = mb[~np.isnan(db)].min(), mb[~np.isnan(db)].max()
print(f'the drift is positive in every bin from beta = {lo_bin:.2f} to {hi_bin:.2f}:', bool(np.all(db[~np.isnan(db)] > 0)))
print(f'drift near the start, beta in [-0.2, 0.2]: {np.mean([b for a, b in zip(mb, db) if abs(a) < 0.2]):+.5f} per step')
print(f'drift around beta = 1: {np.mean([b for a, b in zip(mb, db) if 0.6 < a < 1.4]):+.5f} per step')
print(f'mean commitment drift at the starting value 0.100: {df_[np.argmin(abs(mf - 0.1))]:+.5f} per step')
print(f'mean commitment at the end of the runs: {np.mean([f[-1] for f in TR_F]):.3f}')

B0 = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 12.0]
SB = seeds(12)
T_ADAPT, T_MEAS = (300, 1000) if not QUICK else (60, 200)
far_mean, far_sem = [], []
print(f'drift from a fixed starting preference, every element at beta_0, {len(SB)} seeds, '
      f'measured over {T_MEAS} steps after {T_ADAPT} steps of adaptation')
for b0 in B0:
    d = []
    for sd in SB:
        m = ElementModel(seed=sd); m.beta[:] = b0
        for _ in range(T_ADAPT):
            m.step()
        b1 = m.beta.mean()
        for _ in range(T_MEAS):
            m.step()
        d.append((m.beta.mean() - b1) / T_MEAS)
    far_mean.append(np.mean(d)); far_sem.append(np.std(d, ddof=1) / np.sqrt(len(d)))
    print(f'  beta_0 = {b0:4.1f}: drift {far_mean[-1]:+.5f} +/- {far_sem[-1]:.5f} per step')
sig = [mu > 2 * se for mu, se in zip(far_mean, far_sem)]
fade = next((b0 for i, b0 in enumerate(B0) if not any(sig[i:])), None)
print(f'the drift is significantly positive up to beta_0 = {max([b for b, g in zip(B0, sig) if g], default=None)} '
      f'and not significantly positive from beta_0 = {fade} upward')

30 runs traced. Final mean preference per run: +2.62 +0.72 +3.30 +0.94 +1.86 +2.25 +2.48 +2.00 +1.04 +3.31 +2.40 +3.41 -1.26 +2.54 +4.74 +1.84 +0.18 +0.76 -0.98 +2.25 +1.74 +3.08 +1.95 +1.03 +2.72 -0.56 +1.94 +2.75 +0.97 +0.14
median final preference 1.94, range -1.26 to +4.74, 14 runs above 2 and 5 below 0.5
drift of the mean direction preference over the range the runs reach (per step)
  beta -1.24: +0.00035 +/- 0.00038
  beta -0.83: +0.00015 +/- 0.00035
  beta -0.42: +0.00012 +/- 0.00017
  beta -0.01: +0.00024 +/- 0.00009
  beta +0.41: +0.00040 +/- 0.00012
  beta +0.82: +0.00062 +/- 0.00011
  beta +1.23: +0.00049 +/- 0.00011
  beta +1.64: +0.00035 +/- 0.00012
  beta +2.06: +0.00034 +/- 0.00009
  beta +2.47: +0.00048 +/- 0.00014
  beta +2.88: +0.00031 +/- 0.00014
  beta +3.29: +0.00012 +/- 0.00014
  beta +3.71: -0.00007 +/- 0.00028
  beta +4.12: +0.00026 +/- 0.00026
the drift is positive in every bin from beta = -1.24 to 4.12: False
drift near the start, beta in [-0.2, 0.2]: +0.00024

## Step 3. Figure S7. The map and where each strategy drifts

### The drawing style of the SI figures

In [5]:
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D

SI_RC = {'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.labelsize': 6.6, 'xtick.labelsize': 6.0,
         'ytick.labelsize': 6.0, 'legend.fontsize': 6.0, 'axes.linewidth': 0.6, 'xtick.major.width': 0.6,
         'ytick.major.width': 0.6, 'xtick.major.size': 2.4, 'ytick.major.size': 0, 'legend.frameon': False,
         'axes.spines.top': True, 'axes.spines.right': True, 'mathtext.fontset': 'dejavusans'}
INK, NEUTRAL = '#1A1A1A', '#37474F'
BLUE, RED, GOLD, GREEN, GREY, NAVY = '#2E5090', '#A4161A', '#C9A227', '#2E7D4E', '#8A8A8A', '#1F3F7A'
SI_COND = {'baseline': '#C4C9CD', 'explore': '#8A8A8A', 'imitate': '#5F6B72', 'both': NAVY}
SI_LABEL = {'baseline': 'baseline', 'explore': 'exploration', 'imitate': 'imitation', 'both': 'both'}
_SI_HOT = plt.get_cmap('hot_r')(np.linspace(0.06, 0.94, 256))
_SI_COOL = plt.get_cmap('Blues')(np.linspace(0.92, 0.06, 256))
CMAP3 = LinearSegmentedColormap.from_list('hot_div', np.vstack([_SI_COOL, _SI_HOT]))
HOT3 = LinearSegmentedColormap.from_list('hot_half', CMAP3(np.linspace(0.50, 0.92, 256)))
SI_VLIM = 2.7157337518864986                       # the scale of Fig. 2 and of ref. 3, in bits
SI_CNORM = Normalize(-SI_VLIM, SI_VLIM)
DOT = (0, (1.2, 1.4))


def si_share_colour(w):
    # an element colored by its share against an even share, as in Figs. 2 and 3
    w = np.asarray(w, float); b = np.log2(w / w.sum() * len(w))
    return CMAP3(SI_CNORM(np.where(np.abs(b) < 1e-9, 0.0, b)))


def si_figure(w, h, label):
    plt.rcParams.update(SI_RC)
    fig = plt.figure(figsize=(w, h)); fig.set_label(label)
    return fig


def si_axes(fig, l, b, w, h):
    W, H = fig.get_size_inches()
    return fig.add_axes([l / W, b / H, w / W, h / H])


def si_head(fig, ax, letter, title, dy=0.12):
    # the title centered over the plot, the letter in bold beside its left end
    W, H = fig.get_size_inches()
    bb = ax.get_position()
    y = bb.y1 + dy / H
    t = fig.text((bb.x0 + bb.x1) / 2, y, title, fontsize=7.6, ha='center', va='bottom')
    ext = t.get_window_extent(fig.canvas.get_renderer()).transformed(fig.transFigure.inverted())
    fig.text(ext.x0 - 0.06 / W, y, letter, fontsize=9.0, fontweight='bold', ha='right', va='bottom')


def si_ticks(lo, hi):
    return [lo, (lo + hi) / 2, hi]


def si_nice(lo, hi, zero=False):
    # three round ticks, minimum, middle and maximum, as tight as round numbers allow around the data
    lo, hi = (min(0.0, lo), max(0.0, hi)) if zero else (lo, hi)
    span = (hi - lo) or abs(hi) or 1.0
    for e in range(int(np.floor(np.log10(span))) - 2, int(np.floor(np.log10(span))) + 2):
        for m in (1, 1.5, 2, 2.5, 3, 4, 5, 6, 8):
            u = m * 10.0 ** e
            a = np.floor(lo / u + 1e-9) * u
            b = np.ceil(hi / u - 1e-9) * u
            n = int(round((b - a) / u))
            if n % 2:
                b += u; n += 1
            if n <= 2:
                return [round(a, 10), round(a + u, 10), round(b, 10)]
    return [lo, (lo + hi) / 2, hi]


def si_frame(ax, xt=None, yt=None, xlog=False, ylog=False, xfmt=None, yfmt=None, grid=True):
    # a line above and below, none at the sides, ticks at the given values, a dotted rule at the middle tick
    if xlog:
        ax.set_xscale('log')
    if ylog:
        ax.set_yscale('log')
    g = lambda v, _: f'{v:,g}'
    if xt is not None:
        ax.set_xlim(xt[0], xt[-1]); ax.set_xticks(xt)
        ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(xfmt or g))
        ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
    if yt is not None:
        ax.set_ylim(yt[0], yt[-1]); ax.set_yticks(yt)
        ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(yfmt or g))
        ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
        if grid:
            for v in yt[1:-1]:
                ax.axhline(v, color='gray', alpha=0.45, ls=':', lw=0.5, zorder=0)
    for s_ in ('left', 'right'):
        ax.spines[s_].set_visible(False)
    for s_ in ('top', 'bottom'):
        ax.spines[s_].set_visible(True); ax.spines[s_].set_linewidth(0.7)
    ax.tick_params(axis='y', length=0, pad=1.4)
    ax.tick_params(axis='x', length=2.4, width=0.6, direction='out', pad=1.4)
    ax.set_axisbelow(True)


def si_cbar(fig, l, b, w, h, cmap, lo, hi, label, fmt='{:g}'):
    ax = si_axes(fig, l, b, w, h)
    ax.imshow(np.linspace(1, 0, 128)[:, None], cmap=cmap, aspect='auto', extent=[0, 1, lo, hi])
    ax.set_xticks([]); ax.set_yticks([lo, hi]); ax.set_yticklabels([fmt.format(lo), fmt.format(hi)])
    ax.yaxis.tick_right(); ax.yaxis.set_label_position('right')
    for s_ in ax.spines.values():
        s_.set_visible(True); s_.set_linewidth(0.5); s_.set_linestyle(DOT)
    ax.tick_params(axis='y', length=1.6, pad=1.0, labelsize=5.6)
    ax.set_ylabel(label, fontsize=6.0, labelpad=1.5)
    return ax


def si_bars(ax, groups, colors, labels, dots=True):
    # bars at the median of each group with every run as a dot beside it
    for i, v in enumerate(groups):
        v = np.asarray(v, float)
        ax.bar(i, np.median(v), color=colors[i], width=0.58, zorder=2)
        if dots:
            ax.plot(np.full(len(v), i) + 0.20 + np.linspace(-0.05, 0.05, len(v)), v, '.', ms=1.8, color=INK,
                    alpha=0.6, zorder=3)
    ax.set_xticks(range(len(groups))); ax.set_xticklabels(labels)
    ax.set_xlim(-0.6, len(groups) - 0.4)


def si_bracket(ax, i, j, y, h, text):
    ax.plot([i, i, j, j], [y, y + h, y + h, y], '-', lw=0.6, color=INK)
    ax.text((i + j) / 2, y + h, text, ha='center', va='bottom', fontsize=6.4)


def si_radial(A, n_min=2, dmax=None):
    # elements at the angle of their place on the ring, drawn inward by degree, as in Fig. 5C
    d = A.sum(1).astype(float); n = len(d)
    th = np.pi / 2 - 2 * np.pi * np.arange(n) / n
    dmax = max(d.max() if dmax is None else dmax, n_min + 1)
    r = np.clip(np.log(dmax / np.maximum(d, n_min)) / np.log(dmax / n_min), 0, 1)
    return np.c_[r * np.cos(th), r * np.sin(th)], d


def si_structure(ax, A, W=None, isE=None, s=None, lim=1.12, dmax=None):
    # a learned structure: couplings darker and wider with weight, elements colored by their share of the
    # coupling against an even share and sized by degree, the task-essential elements ringed
    from matplotlib.collections import LineCollection
    xy, d = si_radial(A, dmax=dmax)
    iu = np.array(np.where(np.triu(A))).T
    if W is not None:
        s = (W * A).sum(1) if s is None else s
        ww = W[iu[:, 0], iu[:, 1]]; x = np.sqrt((ww - ww.min()) / max(ww.max() - ww.min(), 1e-12))
    else:
        x = np.full(len(iu), 0.3)
    o = np.argsort(x, kind='stable')
    ax.add_collection(LineCollection(xy[iu[o]], colors=plt.get_cmap('Greys')(0.25 + 0.55 * x[o]),
                                     linewidths=0.12 + 0.8 * x[o], alpha=0.75, zorder=1))
    size = 3.0 + 110.0 * ((d - 1) / (len(d) - 2)) ** 1.15
    o = np.argsort(d, kind='stable')
    ax.scatter(xy[o, 0], xy[o, 1], s=size[o], c=si_share_colour(s)[o], edgecolors=NEUTRAL, linewidths=0.25, zorder=3)
    if isE is not None:
        e = np.where(isE)[0]
        ax.scatter(xy[e, 0], xy[e, 1], s=(np.sqrt(size[e]) + 2.6) ** 2, facecolors='none', edgecolors=INK,
                   linewidths=0.6, zorder=4)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal'); ax.axis('off')

In [6]:
fig = si_figure(6.93, 3.50, 'SI Fig S7')
PW, PH = 1.40, 0.95

def heat(ax, Z, lo, hi):
    ax.imshow(Z, origin='lower', aspect='auto', cmap=HOT3, vmin=lo, vmax=hi, interpolation='nearest')
    ax.set_xticks(range(len(PE2))); ax.set_xticklabels([f'{v:g}' for v in PE2])
    ax.set_yticks(range(len(RHO2))); ax.set_yticklabels([f'{v:g}' for v in RHO2])
    for s_ in ax.spines.values():
        s_.set_visible(True); s_.set_linewidth(0.5); s_.set_linestyle(DOT)
    ax.tick_params(axis='both', length=0, pad=1.4, labelsize=5.6)
    ax.set_xlabel(r'exploration probability $p_e$', labelpad=1); ax.set_ylabel(r'feedback precision $\rho$', labelpad=1)

lo, hi = float(np.floor(DUAL.min() * 10) / 10), float(np.ceil(DUAL.max() * 10) / 10)
ax = si_axes(fig, 0.42, 2.20, PW, PH)
heat(ax, DUAL, lo, hi)
ax.contour(np.arange(len(PE2)), np.arange(len(RHO2)), HUB, levels=[0.2], colors=[NAVY], linewidths=0.9,
           linestyles='--')
ax.plot(PE2.index(0.05), RHO2.index(0.5), 'o', ms=5, mfc='none', mec=INK, mew=0.9)
si_cbar(fig, 0.42 + PW + 0.05, 2.26, 0.05, PH - 0.12, HOT3, lo, hi, 'bits saved', '{:.1f}')
si_head(fig, ax, 'A', 'Compression')

lo2, hi2 = 0.0, float(np.ceil(HUB.max() * 10) / 10)
ax = si_axes(fig, 2.78, 2.20, PW, PH)
heat(ax, HUB, lo2, hi2)
si_cbar(fig, 2.78 + PW + 0.05, 2.26, 0.05, PH - 0.12, HOT3, lo2, hi2, 'share on the essential set', '{:.1f}')
si_head(fig, ax, 'B', 'Share on the essential set')

ax = si_axes(fig, 5.14, 2.20, 1.62, PH)
for rho, col in ((0.0, CMAP3(0.70)), (0.5, CMAP3(0.84)), (1.0, CMAP3(0.95))):
    i = RHO2.index(rho)
    ax.plot(PE2, DUAL[i], 'o-', ms=2.4, lw=0.9, color=col, label=rf'$\rho$ = {rho:g}', clip_on=False, zorder=3)
si_frame(ax, [0, 0.1, 0.2], si_nice(0, DUAL.max(), zero=True))
ax.legend(loc='lower right', handlelength=1.4, labelspacing=0.2, fontsize=5.6, borderaxespad=0.1)
ax.set_xlabel(r'exploration probability $p_e$', labelpad=1); ax.set_ylabel('bits saved', labelpad=1)
si_head(fig, ax, 'C', 'Exploration at three precisions')

ax = si_axes(fig, 0.42, 0.42, 2.80, 0.95)
ok_b = ~np.isnan(db)
ax.axvspan(ends.min(), ends.max(), color=NAVY, alpha=0.06, lw=0, label='where the runs end')
ax.errorbar(mb[ok_b], db[ok_b], yerr=sb[ok_b], fmt='o', ms=2.4, color=NAVY, ecolor=NAVY, elinewidth=0.6,
            capsize=1.4, label=f'inside {len(TR_B)} runs')
ax.errorbar(B0, far_mean, yerr=far_sem, fmt='s', ms=2.4, color=GREY, ecolor=GREY, elinewidth=0.6, capsize=1.4,
            label='from one fixed preference')
xall = np.concatenate([mb[ok_b], np.asarray(B0, float)])
yall = np.concatenate([db[ok_b] + sb[ok_b], db[ok_b] - sb[ok_b], np.asarray(far_mean) + np.asarray(far_sem),
                       np.asarray(far_mean) - np.asarray(far_sem)])
yt = si_nice(yall.min(), yall.max())
si_frame(ax, si_nice(xall.min(), xall.max()), yt, yfmt=lambda v, _: f'{v:.4f}'.rstrip('0').rstrip('.') if v else '0')
ax.axhline(0, color=NEUTRAL, lw=0.6, zorder=1)
ax.legend(loc='lower left', handlelength=1.2, labelspacing=0.2, fontsize=5.6, borderaxespad=0.1)
ax.set_xlabel(r'mean direction preference $\bar{\beta}$', labelpad=1); ax.set_ylabel('drift per step', labelpad=1)
si_head(fig, ax, 'D', 'Where the direction preference goes')

ax = si_axes(fig, 3.96, 0.42, 2.80, 0.95)
ok_f = ~np.isnan(df_)
ax.errorbar(mf[ok_f], df_[ok_f], yerr=sf[ok_f], fmt='o-', ms=2.4, lw=0.8, color=BLUE, elinewidth=0.6, capsize=1.4)
yt = si_nice((df_[ok_f] - sf[ok_f]).min(), (df_[ok_f] + sf[ok_f]).max())
si_frame(ax, si_nice(0, mf[ok_f].max(), zero=True), yt, yfmt=lambda v, _: f'{v:.5f}'.rstrip('0').rstrip('.') if v else '0')
ax.axhline(0, color=NEUTRAL, lw=0.6, zorder=1)
ax.axvline(0.1, ls=':', color=INK, lw=0.7)
ax.text(0.1, 0.9 * yt[-1], 'start ', fontsize=5.6, ha='right', va='top')
ax.set_xlabel(r'mean commitment $\bar{f}$', labelpad=1); ax.set_ylabel('drift per step', labelpad=1)
si_head(fig, ax, 'E', 'Where commitment goes')
plt.show()